# 12 · `gl_engine/interp/interpreter.py`

## What this file is for

The machine that runs ISO's rules: **frames, dispatch, lookup, rounding, and the trace.**

[`11-interp-nodes`](11-interp-nodes.ipynb) is the vocabulary; this is the thing that walks a rule and evaluates it. Four of its five jobs are ordinary interpreter work. The fifth — the trace — is why an underwriter can be told *why* a premium is what it is.

**Depends on:** [`10-interp-program`](10-interp-program.ipynb), [`11-interp-nodes`](11-interp-nodes.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.interp import interpreter

for name, obj in vars(interpreter).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != interpreter.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Build an interpreter, evaluate something, look at what it recorded.

In [ ]:
import xml.etree.ElementTree as ET
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook
from gl_engine.interp.interpreter import Interpreter, Frame, MAX_DEPTH, ROUNDING_MODES
from gl_engine.interp.program import Program
from gl_engine.interp import tree as T

NS = 'xmlns="http://www.verisk.com/iso/erc/Rule"'

book  = ResolvedBook(EditionResolver().resolve("GA", "20260811"))
ip    = Interpreter(book)
frame = Frame(data=T.Node.from_dict("GeneralLiability", {"StateCode": "GA"}),
              program=Program(book.parent.package),
              rule_file="GeneralLiabilityRules")

print("rounding mode :", ip.rounding if hasattr(ip, "rounding") else "(default)")
print("modes offered :", list(ROUNDING_MODES))
print("max call depth:", MAX_DEPTH)

## The interesting case

### Rounding, and the oldest open question in the project

`@DecimalPlaces` appears 7,682 times in ISO's content. **Nowhere does ISO state which rounding rule to apply.** The corpus does not settle it — so it was settled against ISO's live service instead, and the answer was *round, don't truncate*.

Half-up versus half-even is still open, because a true tie is astonishingly rare: of 1,529 rounding operations across every sample, exactly one lands on an exact `.5`.

In [ ]:
from decimal import Decimal

el = ET.fromstring(f'<Constant {NS} DecimalPlaces="0"/>')

for mode in ROUNDING_MODES:
    got = Interpreter(book, rounding=mode).round_to(Decimal("2.5"), el, "demo")
    print(f"{mode:<17} 2.5 -> {got}")

print()
print("ROUND_DOWN is settled: ISO rounds. It changed the premium in 37 of 51")
print("jurisdictions, and ISO agreed with rounding in all 50 compared.")
print("HALF_UP vs HALF_EVEN differ only on an exact tie -- still open (OI-70).")

### The trace is not logging

Every lookup, write and rounding decision is recorded as it happens. This is what the interface renders, and it is what made the engine debuggable while it was being built.

In [ ]:
print("trace entries after a bare eval:", len(ip.trace))

lit = ET.fromstring(f'<Constant {NS} Type="decimal">1.5</Constant>')
ip.eval(lit, frame)

for entry in ip.trace[:8]:
    print("  ", entry)
if not ip.trace:
    print("   (a Constant writes nothing -- see notebook 16 for a full rating trace)")

### A frame is the scope

Rules are called with parameters, and a called rule sees its own scope. `depth` is what stops a cyclic rule set running forever.

In [ ]:
print("depth at entry :", frame.depth)
child = frame.with_params({"SomeParam": "1"})
print("has_param      :", child.has_param("SomeParam"), "->", child.param("SomeParam"))
print("parent has it  :", frame.has_param("SomeParam"))
print()
print(f"MAX_DEPTH is {MAX_DEPTH}: a rule set that calls itself forever stops,")
print("rather than exhausting the stack somewhere unrecognisable.")

## What it refuses

Two refusals, and both exist so that a wrong number is impossible rather than unlikely.

In [ ]:
from gl_engine.interp.values import InterpretError

try:
    ip.eval(ET.fromstring(f'<NotAnInstruction {NS}/>'), frame)
except InterpretError as e:
    print("unknown instruction:", str(e).split("--")[0].strip())

try:
    ip.lookup("NoSuchTable", "Factor", ["GA"], "Exact", "decimal", "demo")
except Exception as e:
    print(f"missing table      : {type(e).__name__}: {str(e)[:80]}")

A **lookup miss** is different from a missing table, and the distinction is deliberate: a miss records itself in the trace and returns null, so ISO's own rules can retry against countrywide. That retry is ISO's content, not ours — grep this package for `'CW'` and you will find nothing.

## Try it yourself

1. Grep `gl_engine/interp/` for `"CW"`. What does the absence prove about where the state/countrywide fallback lives?
2. Set `trace=False` and rate something. What does the engine lose, and what does it gain?
3. Find `trace_exhausted`. When does the interpreter decide it has recorded enough?

In [ ]:
# your turn